# Compute the candidate space

## Code introduction

Dear reader,

This script is supporting chapter 7.2.3 of my MSc thesis. It computes the upper bound of the number of candidate flexibility-use patterns. Based on the target week and configuration settings in the first cell block, the script computes the total number of candidate patterns that could be produced in theory, if there were no computational safety cap added by "max_completed_patterns_to_score".

Best,

Yad Khan

## Script

In [1]:
# Import libraries, packages and input data

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Iterable, Optional, Any, Set
import itertools
import math
import json
import warnings

import numpy as np #for numerical calculations
import pandas as pd #for loading CSVs,
import matplotlib.pyplot as plt


# =============================================================================
# 1. IMPORTS AND CONFIGURATION
# =============================================================================

@dataclass(frozen=True)
class Config:
    """Configuration values that define the reduced case-study implementation."""

    dataset1_path: Path = Path("/Users/yadkhan/Downloads/Case study files/Dataset1_charging_reports.csv")
    dataset3_path: Path = Path("/Users/yadkhan/Downloads/Case study files/Dataset3_session_predictions.csv")
    dataset4_path: Path = Path("/Users/yadkhan/Downloads/Case study files/Dataset4_hourly_predictions.csv")

    output_dir: Path = Path("outputs_dbn_week_2019-07-08")

    # Thesis case-study scope: one location and one target week inside 2019.
    location: str = "TRO_R"
    target_week_start: str = "2019-06-17 00:00:00"
    target_week_days: int = 5

    # Weekday evening target window from the thesis: Monday-Friday, 16:00-22:00.
    target_start_hour: int = 16 
    target_end_hour_exclusive: int = 22

    # The thesis states that B_{i,t-1} approximately zero means [-0.05, 0.05] kWh.
    balance_zero_tolerance_kwh: float = 0.05

    # Numerical tolerance used when checking target/energy constraints after inference.
    feasibility_tolerance_kwh: float = 0.05

    # This is an implementation guard. For the selected week, reduction enumeration is exact.
    # Addition completion is searched by dynamic programming and keeps the best solutions per session.
    # This value can be increased if broader candidate set is desired; runtime and output size will increase accordingly
    max_addition_solutions_per_session: int = 5

    # Safety cap only to prevent accidental memory blow-up if a different week is used.
    max_completed_patterns_to_score: int = 200000


CONFIG = Config()

In [3]:
# =============================================================================
# 2. FILE PATHS
# =============================================================================

# Edit these paths if your files have different names. If the script is run from the
# same folder as the CSV files, the defaults above are enough.
FILE_PATHS = {
    "dataset1": CONFIG.dataset1_path,
    "dataset3": CONFIG.dataset3_path,
    "dataset4": CONFIG.dataset4_path,
}


In [5]:
# =============================================================================
# 3. DATA LOADING
# =============================================================================

def resolve_input_path(path: Path) -> Path:
    """
    Resolve an input path.
    """
    if path.exists():
        return path
    matches = sorted(Path(".").glob(path.stem + "*.csv"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find input file: {path.resolve()}")


def read_semicolon_decimal_csv(path: Path) -> pd.DataFrame:
    """
    Read the Sørensen et al. CSV format.

    The files are semicolon-separated and use comma decimals. 'NA' is treated as missing.
    """
    resolved = resolve_input_path(path)
    return pd.read_csv(
        resolved,
        sep=";",
        decimal=",",
        quotechar='"',
        na_values=["NA", ""],
        keep_default_na=True,
    )


def load_all_data(paths: Dict[str, Path]) -> Dict[str, pd.DataFrame]:
    """
    Load the datasets retained in the reduced model scope.
    """
    return {name: read_semicolon_decimal_csv(path) for name, path in paths.items()}


In [7]:
# =============================================================================
# 4. SCHEMA VALIDATION
# =============================================================================

REQUIRED_COLUMNS = {
    "dataset1": [
        "location", "user_id", "session_id", "plugin_time", "plugout_time",
        "connection_time", "energy_session",
    ],
    "dataset3": [
        "user_id", "session_id", "charging_time", "SoC_diff", "SoC_start",
        "idle_time", "idle_session", "non_flex_session",
    ],
    "dataset4": [
        "user_id", "session_id", "date_from", "energy_charged_i",
        "energy_idle_i", "energy_connected_i", "SoC_diff_i", "SoC_from_i", "SoC_to_i",
    ],
}


def validate_required_columns(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Check whether all expected columns are present."""
    rows = []
    for name, required in REQUIRED_COLUMNS.items():
        df = data[name]
        missing = [c for c in required if c not in df.columns]
        rows.append({
            "dataset": name,
            "n_rows": len(df),
            "n_columns": len(df.columns),
            "missing_required_columns": ", ".join(missing),
            "schema_ok": len(missing) == 0,
        })
    summary = pd.DataFrame(rows)
    if not summary["schema_ok"].all():
        raise ValueError("At least one dataset is missing required columns. See schema summary.")
    return summary


def validate_basic_data_quality(data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Create a missing-value overview for the required columns."""
    rows = []
    for name, required in REQUIRED_COLUMNS.items():
        df = data[name]
        for col in required:
            rows.append({
                "dataset": name,
                "column": col,
                "missing_count": int(df[col].isna().sum()),
                "missing_share": float(df[col].isna().mean()),
            })
    return pd.DataFrame(rows)


In [9]:
# =============================================================================
# 5. PREPROCESSING
# =============================================================================

def parse_time_columns(data: Dict[str, pd.DataFrame]) -> None:
    """Parse the timestamp columns used by Chapter 6 to define session windows and hourly time."""
    data["dataset1"]["plugin_time"] = pd.to_datetime(data["dataset1"]["plugin_time"], errors="coerce")
    data["dataset1"]["plugout_time"] = pd.to_datetime(data["dataset1"]["plugout_time"], errors="coerce")
    data["dataset4"]["date_from"] = pd.to_datetime(data["dataset4"]["date_from"], errors="coerce")


def validate_hourly_time_format(dataset4: pd.DataFrame) -> None:
    """Check that Dataset 4 is hourly, as required by Chapter 6."""
    if dataset4["date_from"].isna().any():
        raise ValueError("Dataset 4 contains unparseable date_from timestamps.")
    non_hourly = dataset4[dataset4["date_from"].dt.minute.ne(0) | dataset4["date_from"].dt.second.ne(0)]
    if len(non_hourly) > 0:
        raise ValueError(f"Dataset 4 contains {len(non_hourly)} non-hourly date_from values.")


def select_target_week_scope(data: Dict[str, pd.DataFrame], config: Config) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DatetimeIndex, pd.Timestamp, pd.Timestamp]:
    """
    Select the one-week reduced case-study scope.

    The selected sessions are TRO_R sessions whose physical connection window intersects the
    target week, and whose full connection window stays inside 2019. This avoids the 2018/2020
    boundary issue and preserves the Chapter 6 logic that shifting happens inside T_i.
    """
    d1 = data["dataset1"].copy()
    d3 = data["dataset3"].copy()
    d4 = data["dataset4"].copy()

    week_start = pd.Timestamp(config.target_week_start)
    week_end = week_start + pd.Timedelta(days=config.target_week_days)

    # Merge Dataset 1 session windows with Dataset 3 session-level flexibility variables.
    session_level = d1[d1["location"].eq(config.location)].merge(
        d3[["session_id", "charging_time", "idle_time", "non_flex_session"]],
        on="session_id",
        how="left",
    )

    # Sessions that intersect the target week and stay inside 2019.
    sessions = session_level[
        (session_level["plugin_time"] < week_end)
        & (session_level["plugout_time"] >= week_start)
        & (session_level["plugin_time"] >= pd.Timestamp("2019-01-01 00:00:00"))
        & (session_level["plugout_time"] < pd.Timestamp("2020-01-01 00:00:00"))
    ].copy()

    required_session_cols = ["plugin_time", "plugout_time", "connection_time", "charging_time", "idle_time"]
    missing_required = sessions[required_session_cols].isna().sum()
    if missing_required.sum() > 0:
        raise ValueError(
            "Selected target week has missing required session-level values. "
            f"Missing counts: {missing_required.to_dict()}"
        )

    # Chapter 6 treats sessions with idle time below one hour as non-flexible.
    # Dataset 3 also flags non-flex sessions via non_flex_session; both definitions agree in this data.
    sessions["is_flexible"] = sessions["idle_time"].ge(1.0) & sessions["non_flex_session"].isna()

    # Dataset 4 rows for the selected sessions. We keep the full 2019 rows for these sessions,
    # because energy can be shifted inside each full connection window T_i.
    hourly = d4[d4["session_id"].isin(sessions["session_id"])].copy()
    hourly = hourly[
        (hourly["date_from"] >= pd.Timestamp("2019-01-01 00:00:00"))
        & (hourly["date_from"] < pd.Timestamp("2020-01-01 00:00:00"))
    ].copy()

    if hourly[["user_id", "session_id", "date_from", "energy_connected_i"]].isna().any().any():
        raise ValueError("Selected hourly scope has missing identifiers, timestamps, or energy_connected_i.")

    # Appendix A says e_obs and e_idle equal zero when the numerical value is zero or missing.
    # Therefore NA in energy_charged_i and energy_idle_i is not deleted here; it is mapped to zero.
    hourly["energy_charged_raw"] = hourly["energy_charged_i"]
    hourly["energy_idle_raw"] = hourly["energy_idle_i"]
    hourly["energy_charged_i"] = hourly["energy_charged_i"].fillna(0.0)
    hourly["energy_idle_i"] = hourly["energy_idle_i"].fillna(0.0)

    hourly = hourly.merge(
        sessions[["session_id", "is_flexible", "idle_time", "non_flex_session", "plugin_time", "plugout_time"]],
        on="session_id",
        how="left",
    )

    # Define W: target weekday-evening hours in the selected week.
    all_week_hours = pd.date_range(week_start, week_end - pd.Timedelta(hours=1), freq="h")
    target_hours = pd.DatetimeIndex([
        ts for ts in all_week_hours
        if ts.weekday() < 5 and config.target_start_hour <= ts.hour < config.target_end_hour_exclusive
    ])

    return sessions, hourly, target_hours, week_start, week_end


In [11]:
# =============================================================================
# 6. STATE DISCRETISATION
# =============================================================================

ENERGY_STATE_THRESHOLDS = {
    # These are the fixed Appendix A / Table 6.3 state thresholds.
    "e_obs": {"q33": 3.242339178, "q67": 4.521205382},
    "e_idle": {"q33": 3.579171, "q67": 6.260996},
    "e_conn": {"q33": 3.608115, "q67": 6.710458},
}

F_REQ_STATES = [
    "large reduction",
    "small reduction",
    "unchanged",
    "small addition",
    "large addition",
]

ENERGY_STATES = ["zero", "low", "medium", "high"]
LOAD_STATES = ["Within cap", "Exceeds cap"]
TARGET_STATES = [0, 1]


def discretise_energy_value(value: float, quantity: str) -> str:
    """Apply the deterministic CPT/discretisation for e_obs, e_idle, or e_conn."""
    if pd.isna(value) or abs(float(value)) < 1e-12:
        return "zero"
    if value < 0:
        raise ValueError(f"Negative value not allowed for {quantity}: {value}")
    q33 = ENERGY_STATE_THRESHOLDS[quantity]["q33"]
    q67 = ENERGY_STATE_THRESHOLDS[quantity]["q67"]
    if 0 < value <= q33:
        return "low"
    if q33 < value <= q67:
        return "medium"
    return "high"


def add_discrete_state_columns(hourly: pd.DataFrame) -> pd.DataFrame:
    """Add DBN state labels for the three input nodes."""
    hourly = hourly.copy()
    hourly["e_obs_state"] = hourly["energy_charged_i"].apply(lambda x: discretise_energy_value(x, "e_obs"))
    hourly["e_idle_state"] = hourly["energy_idle_i"].apply(lambda x: discretise_energy_value(x, "e_idle"))
    hourly["e_conn_state"] = hourly["energy_connected_i"].apply(lambda x: discretise_energy_value(x, "e_conn"))
    return hourly


In [13]:
# =============================================================================
# 7. CPT CONSTRUCTION FROM APPENDIX A
# =============================================================================

def balance_state(balance_kwh: float, zero_tolerance: float) -> str:
    """Map B_{i,t-1} to the three balance conditions used in Appendix A."""
    if balance_kwh < -zero_tolerance:
        return "negative"  # B_{i,t-1} < 0
    if balance_kwh > zero_tolerance:
        return "positive"  # B_{i,t-1} > 0
    return "zero"          # B_{i,t-1} approximately 0


def f_req_cpt_probs(e_obs_state: str, e_idle_state: str, e_conn_state: str, b_state: str) -> Dict[str, float]:
    """
    Appendix A CPT for f_req.

    The probabilities are implemented following the CPTs in Appendix A and the logic described in chapter 6:
    - no connection/no energy possibility -> unchanged with probability 1;
    - only idle capacity -> addition/unchanged probabilities;
    - only observed charging -> reduction/unchanged probabilities;
    - both observed and idle -> all five states possible, with B deciding the preference.
    """
    obs_pos = e_obs_state != "zero"
    idle_pos = e_idle_state != "zero"
    conn_pos = e_conn_state != "zero"

    if (not conn_pos) or ((not obs_pos) and (not idle_pos)):
        probs = [0.00, 0.00, 1.00, 0.00, 0.00]
    elif (not obs_pos) and idle_pos:
        if b_state == "negative":
            probs = [0.00, 0.00, 0.20, 0.35, 0.45]
        elif b_state == "zero":
            probs = [0.00, 0.00, 0.50, 0.25, 0.25]
        else:
            probs = [0.00, 0.00, 0.85, 0.10, 0.05]
    elif obs_pos and (not idle_pos):
        if b_state == "negative":
            probs = [0.05, 0.10, 0.85, 0.00, 0.00]
        elif b_state == "zero":
            probs = [0.25, 0.25, 0.50, 0.00, 0.00]
        else:
            probs = [0.45, 0.35, 0.20, 0.00, 0.00]
    else:
        if b_state == "negative":
            probs = [0.02, 0.03, 0.10, 0.35, 0.50]
        elif b_state == "zero":
            probs = [0.20, 0.20, 0.20, 0.20, 0.20]
        else:
            probs = [0.50, 0.35, 0.10, 0.03, 0.02]

    return dict(zip(F_REQ_STATES, probs))


def validate_cpts() -> pd.DataFrame:
    """Validate that all implemented CPT rows sum to one."""
    rows = []

    # Deterministic input-node CPTs.
    for quantity in ["e_obs", "e_idle", "e_conn"]:
        for state in ENERGY_STATES:
            rows.append({
                "cpt": quantity,
                "parent_state": state,
                "probability_sum": 1.0,
                "valid": True,
            })

    # Hidden-node CPT.
    for e_obs in ENERGY_STATES:
        for e_idle in ENERGY_STATES:
            for e_conn in ENERGY_STATES:
                for b in ["negative", "zero", "positive"]:
                    probs = f_req_cpt_probs(e_obs, e_idle, e_conn, b)
                    psum = sum(probs.values())
                    rows.append({
                        "cpt": "f_req",
                        "parent_state": f"e_obs={e_obs}, e_idle={e_idle}, e_conn={e_conn}, B={b}",
                        "probability_sum": psum,
                        "valid": abs(psum - 1.0) < 1e-12,
                    })

    # Deterministic output CPTs for tilde L and X.
    rows.append({"cpt": "tilde_L", "parent_state": "adjusted load <= C90", "probability_sum": 1.0, "valid": True})
    rows.append({"cpt": "tilde_L", "parent_state": "adjusted load > C90", "probability_sum": 1.0, "valid": True})
    rows.append({"cpt": "X", "parent_state": "tilde_L=Within cap", "probability_sum": 1.0, "valid": True})
    rows.append({"cpt": "X", "parent_state": "tilde_L=Exceeds cap", "probability_sum": 1.0, "valid": True})

    validation = pd.DataFrame(rows)
    if not validation["valid"].all():
        raise ValueError("At least one CPT row does not sum to 1.")
    return validation


In [15]:
# =============================================================================
# 8. DBN / TIME-SLICED MODEL CONSTRUCTION
# =============================================================================

def build_node_state_table() -> pd.DataFrame:
    """Return the DBN node-state table used by the implementation."""
    return pd.DataFrame([
        {"thesis_symbol": "e_obs_i_t", "python_name": "e_obs_state", "role": "input", "status": "observed", "states": ENERGY_STATES},
        {"thesis_symbol": "e_idle_i_t", "python_name": "e_idle_state", "role": "input", "status": "derived", "states": ENERGY_STATES},
        {"thesis_symbol": "e_conn_i_t", "python_name": "e_conn_state", "role": "input", "status": "derived", "states": ENERGY_STATES},
        {"thesis_symbol": "f_req_i_t", "python_name": "f_req_state", "role": "hidden", "status": "inferred", "states": F_REQ_STATES},
        {"thesis_symbol": "tilde_L_t", "python_name": "adjusted_load_state", "role": "output", "status": "derived deterministic", "states": LOAD_STATES},
        {"thesis_symbol": "X_t", "python_name": "target_state", "role": "output/evidence", "status": "derived deterministic", "states": TARGET_STATES},
    ])


def build_edge_list() -> pd.DataFrame:
    """
    Return the parent-child structure implemented as local factors.

    B_{i,t-1} is listed as a deterministic external parent/context variable, not as a DBN node.
    """
    edges = [
        ("e_obs_i_t", "f_req_i_t", "same session-hour input state"),
        ("e_idle_i_t", "f_req_i_t", "same session-hour input state"),
        ("e_conn_i_t", "f_req_i_t", "same session-hour input state"),
        ("B_i_t_minus_1", "f_req_i_t", "external deterministic balance context"),
        ("f_req_i_t", "delta_i_t", "state-to-midpoint deterministic mapping"),
        ("delta_i_t", "B_i_t", "external deterministic balance update"),
        ("delta_i_t", "tilde_L_t", "aggregate adjusted load contribution"),
        ("L_obs_t", "tilde_L_t", "observed aggregate load baseline"),
        ("C90", "tilde_L_t", "target cap threshold"),
        ("tilde_L_t", "X_t", "deterministic target-state mapping"),
    ]
    return pd.DataFrame(edges, columns=["parent", "child", "implementation_note"])


In [25]:
# =============================================================================
# 9. EVIDENCE CONSTRUCTION
# =============================================================================

def compute_observed_target_loads(hourly: pd.DataFrame, target_hours: pd.DatetimeIndex, week_start: pd.Timestamp, week_end: pd.Timestamp) -> Tuple[pd.Series, float, List[pd.Timestamp]]:
    """
    Compute L_obs_t and C90 for the selected target week.

    C90 is not hard-coded. It is calculated from the selected target-week weekday-evening loads.
    """
    week_rows = hourly[(hourly["date_from"] >= week_start) & (hourly["date_from"] < week_end)]
    load_by_hour = week_rows.groupby("date_from")["energy_charged_i"].sum()
    target_loads = pd.Series([float(load_by_hour.get(ts, 0.0)) for ts in target_hours], index=target_hours, name="L_obs_t")
    c90 = float(target_loads.quantile(0.90))
    exceed_hours = list(target_loads[target_loads > c90 + 1e-12].index)
    return target_loads, c90, exceed_hours


def build_target_evidence(target_hours: pd.DatetimeIndex) -> pd.DataFrame:
    """The target evidence is X_t = 0 for every weekday-evening target hour."""
    return pd.DataFrame({"date_from": target_hours, "X_t_evidence": 0})



In [27]:


# =============================================================================
# 10. INFERENCE
# =============================================================================

def delta_from_f_req_state(f_state: str, e_obs_kwh: float, e_idle_kwh: float) -> float:
    """Equation 6.23 midpoint mapping from f_req state to delta_i_t."""
    if f_state == "large reduction":
        return -0.75 * e_obs_kwh
    if f_state == "small reduction":
        return -0.25 * e_obs_kwh
    if f_state == "unchanged":
        return 0.0
    if f_state == "small addition":
        return 0.25 * e_idle_kwh
    if f_state == "large addition":
        return 0.75 * e_idle_kwh
    raise ValueError(f"Unknown f_req state: {f_state}")


class FactorBasedWeekInference:
    """
    Custom factor-based inference engine for the selected target week.

    This class keeps the thesis DBN logic but does not force B_i_t into a DBN package. Instead,
    it scores candidate F_req patterns by multiplying the local Appendix A CPT probabilities while
    updating B_i_t deterministically through time.

    Important implementation convention:
    A candidate pattern dictionary only stores non-unchanged choices explicitly. During full scoring,
    every omitted flexible session-hour is explicitly treated as f_req_i_t = "unchanged". This matches
    the thesis meaning of F_req as the full indexed collection over all i in I_f and t in T_i.
    """

    def __init__(
        self,
        hourly: pd.DataFrame,
        target_hours: pd.DatetimeIndex,
        week_start: pd.Timestamp,
        week_end: pd.Timestamp,
        c90: float,
        target_loads: pd.Series,
        exceed_hours: List[pd.Timestamp],
        config: Config,
    ) -> None:
        self.hourly = hourly.copy()
        self.target_hours = pd.DatetimeIndex(target_hours)
        self.target_hours_set = set(self.target_hours)
        self.week_start = week_start
        self.week_end = week_end
        self.c90 = c90
        self.target_loads = target_loads
        self.exceed_hours = exceed_hours
        self.config = config

        # Store every session-hour row as the numerical and discretised context needed for CPT lookup.
        self.row_records: Dict[Tuple[str, pd.Timestamp], Dict[str, Any]] = {}
        for _, r in self.hourly.iterrows():
            key = (str(r["session_id"]), pd.Timestamp(r["date_from"]))
            self.row_records[key] = {
                "session_id": str(r["session_id"]),
                "date_from": pd.Timestamp(r["date_from"]),
                "eobs": float(r["energy_charged_i"]),
                "eidle": float(r["energy_idle_i"]),
                "econn": float(r["energy_connected_i"]),
                "obs_state": r["e_obs_state"],
                "idle_state": r["e_idle_state"],
                "conn_state": r["e_conn_state"],
                "is_flexible": bool(r["is_flexible"]),
            }

        # session_keys defines the full domain of F_req for the selected implementation scope:
        # all flexible sessions i in I_f and all connected hourly rows t in T_i.
        self.session_keys: Dict[str, List[Tuple[str, pd.Timestamp]]] = {}
        flexible_rows = self.hourly[self.hourly["is_flexible"]].sort_values(["session_id", "date_from"])
        for sid, grp in flexible_rows.groupby("session_id"):
            self.session_keys[str(sid)] = [(str(sid), pd.Timestamp(t)) for t in grp["date_from"].tolist()]

        self.all_flexible_keys: List[Tuple[str, pd.Timestamp]] = [
            key for sid in sorted(self.session_keys) for key in self.session_keys[sid]
        ]
        self.all_flexible_key_set: Set[Tuple[str, pd.Timestamp]] = set(self.all_flexible_keys)

        # L_obs_t baseline for the selected week; tilde_L_t is produced by adding deltas to this baseline.
        week_rows = self.hourly[(self.hourly["date_from"] >= self.week_start) & (self.hourly["date_from"] < self.week_end)]
        self.week_base_load: Dict[pd.Timestamp, float] = week_rows.groupby("date_from")["energy_charged_i"].sum().to_dict()

    def f_state_for_key(self, pattern: Dict[Tuple[str, pd.Timestamp], str], key: Tuple[str, pd.Timestamp]) -> str:
        """
        Return the candidate f_req state for one session-hour.

        The candidate dictionary stores only explicit reductions/additions. If a key is not present,
        the implementation treats that session-hour as "unchanged". This is not a new assumption:
        it is the operational meaning of no charging adjustment, i.e. delta_i_t = 0.
        """
        return pattern.get(key, "unchanged")

    def score_pattern(
        self,
        pattern: Dict[Tuple[str, pd.Timestamp], str],
        relevant_sessions: Optional[Set[str]] = None,
    ) -> Tuple[float, Optional[Dict[str, float]]]:
        """
        Score a candidate F_req pattern by multiplying local CPT probabilities.

        If relevant_sessions is None, the score is computed over the full thesis domain:
        all flexible sessions i in I_f and all connected hours t in T_i.

        If relevant_sessions is provided, the same scoring logic is applied only to those sessions.
        This restricted mode is used internally during candidate generation to identify session-level
        debts that must be repaid. It is not used for the final posterior score.
        """
        if relevant_sessions is None:
            relevant_sessions = set(self.session_keys.keys())

        logp = 0.0
        balances: Dict[str, float] = {}
        tol = self.config.balance_zero_tolerance_kwh

        for sid in sorted(relevant_sessions):
            balance = 0.0
            for key in self.session_keys.get(sid, []):
                rec = self.row_records[key]

                # Unassigned flexible session-hours are explicitly evaluated as unchanged.
                f_state = self.f_state_for_key(pattern, key)

                # The Appendix A CPT uses the previous cumulative balance as external context.
                b_state = balance_state(balance, tol)
                probs = f_req_cpt_probs(rec["obs_state"], rec["idle_state"], rec["conn_state"], b_state)
                p = probs[f_state]

                # A zero-probability local CPT choice makes the full candidate impossible.
                if p <= 0:
                    return -math.inf, None

                logp += math.log(p)

                # Apply the deterministic delta mapping and update B_i_t before the next hour.
                balance += delta_from_f_req_state(f_state, rec["eobs"], rec["eidle"])
            balances[sid] = balance

        return logp, balances

    def make_pattern_detail_records(
        self,
        pattern_id: int,
        pattern: Dict[Tuple[str, pd.Timestamp], str],
    ) -> List[Dict[str, Any]]:
        """
        Expand one candidate pattern into full session-hour detail rows.

        This is used after Step 11 has been corrected: each candidate pattern is interpreted as a full
        F_req realisation. Therefore every flexible (i,t) is written to the detail table, including omitted
        session-hours that are treated as unchanged.
        """
        rows: List[Dict[str, Any]] = []
        tol = self.config.balance_zero_tolerance_kwh

        for sid in sorted(self.session_keys):
            balance = 0.0
            for key in self.session_keys[sid]:
                rec = self.row_records[key]
                f_state = self.f_state_for_key(pattern, key)
                b_prev = balance
                b_state = balance_state(b_prev, tol)
                probs = f_req_cpt_probs(rec["obs_state"], rec["idle_state"], rec["conn_state"], b_state)
                local_p = probs[f_state]
                delta = delta_from_f_req_state(f_state, rec["eobs"], rec["eidle"])
                b_after = b_prev + delta

                rows.append({
                    "pattern_id": pattern_id,
                    "session_id": sid,
                    "date_from": key[1],
                    "f_req_state": f_state,
                    "delta_kwh": delta,
                    "e_obs_kwh": rec["eobs"],
                    "e_idle_kwh": rec["eidle"],
                    "e_conn_kwh": rec["econn"],
                    "e_obs_state": rec["obs_state"],
                    "e_idle_state": rec["idle_state"],
                    "e_conn_state": rec["conn_state"],
                    "B_prev_kwh": b_prev,
                    "B_prev_state": b_state,
                    "local_cpt_probability": local_p,
                    "B_after_kwh": b_after,
                    "explicitly_assigned_in_candidate": key in pattern,
                    "is_adjusted": f_state != "unchanged",
                })

                balance = b_after

        return rows

    def adjusted_load_from_pattern(self, pattern: Dict[Tuple[str, pd.Timestamp], str]) -> Dict[pd.Timestamp, float]:
        """Compute tilde_L_t by adding all explicit delta_i_t values to L_obs_t."""
        adjusted = dict(self.week_base_load)
        for key, f_state in pattern.items():
            sid, ts = key
            if self.week_start <= ts < self.week_end:
                rec = self.row_records[key]
                delta = delta_from_f_req_state(f_state, rec["eobs"], rec["eidle"])
                adjusted[ts] = adjusted.get(ts, 0.0) + delta
        return adjusted

    def reduction_options_for_key(self, key: Tuple[str, pd.Timestamp]) -> List[Tuple[str, float]]:
        """Allowed reduction options for a flexible charged session-hour in an exceeded target hour."""
        rec = self.row_records[key]
        eobs = rec["eobs"]
        return [
            ("unchanged", 0.0),
            ("small reduction", delta_from_f_req_state("small reduction", eobs, 0.0)),
            ("large reduction", delta_from_f_req_state("large reduction", eobs, 0.0)),
        ]

    def enumerate_reduction_patterns(self) -> List[Dict[Tuple[str, pd.Timestamp], str]]:
        """
        Enumerate reduction combinations for all exceeded target hours.

        For the chosen week, this is exact over the reduction choices at the exceeded target hours:
        unchanged, small reduction, or large reduction for each flexible session-hour that is actively
        charging in the exceeded hour.
        """
        patterns: List[Dict[Tuple[str, pd.Timestamp], str]] = [dict()]

        for ts in self.exceed_hours:
            needed_reduction = float(self.target_loads.loc[ts] - self.c90)
            active = self.hourly[
                (self.hourly["date_from"].eq(ts))
                & (self.hourly["is_flexible"])
                & (self.hourly["energy_charged_i"].gt(0.0))
            ]
            active_keys = [(str(r["session_id"]), pd.Timestamp(r["date_from"])) for _, r in active.iterrows()]

            hour_patterns: List[Dict[Tuple[str, pd.Timestamp], str]] = []
            for combo in itertools.product(*[self.reduction_options_for_key(k) for k in active_keys]):
                total_reduction = -sum(delta for _, delta in combo)

                # Keep only reduction combinations that make this exceeded target hour feasible.
                if total_reduction >= needed_reduction - 1e-12:
                    candidate = {}
                    for key, (state, _) in zip(active_keys, combo):
                        if state != "unchanged":
                            candidate[key] = state
                    hour_patterns.append(candidate)

            if not hour_patterns:
                raise RuntimeError(f"No reduction pattern can satisfy exceeded hour {ts}.")

            new_patterns: List[Dict[Tuple[str, pd.Timestamp], str]] = []
            for base in patterns:
                for hour_pattern in hour_patterns:
                    merged = dict(base)
                    ok = True
                    for key, state in hour_pattern.items():
                        if key in merged and merged[key] != state:
                            ok = False
                            break
                        merged[key] = state
                    if ok:
                        new_patterns.append(merged)
            patterns = new_patterns

        return patterns

    def find_additions_for_session(
        self,
        sid: str,
        debt_kwh: float,
        base_pattern: Dict[Tuple[str, pd.Timestamp], str],
        current_adjusted_load: Dict[pd.Timestamp, float],
    ) -> List[Tuple[float, int, float, Tuple[Tuple[Tuple[str, pd.Timestamp], str, float], ...]]]:
        """
        Find addition states that repay a session's negative balance.

        This dynamic-programming search uses the thesis midpoint deltas and the 0.05 kWh balance
        tolerance. It also prevents additions that would violate X_t = 0 in target hours.
        """
        candidates: List[Tuple[Tuple[str, pd.Timestamp], str, float]] = []
        for key in self.session_keys.get(sid, []):
            if key in base_pattern:
                continue
            rec = self.row_records[key]
            if rec["eidle"] <= 0:
                continue

            # Addition candidates use available idle capacity only.
            for state in ["small addition", "large addition"]:
                delta = delta_from_f_req_state(state, rec["eobs"], rec["eidle"])
                if delta <= 0:
                    continue
                ts = key[1]

                # Shifted energy may only be added back inside the selected target week.
                if not (self.week_start <= ts < self.week_end):
                    continue

                # Do not repair one session by creating a new target-hour cap violation.
                if ts in self.target_hours_set and current_adjusted_load.get(ts, 0.0) + delta > self.c90 + 1e-12:
                    continue
                candidates.append((key, state, delta))

        scale = 100  # centi-kWh rounding for DP bookkeeping
        target = int(round(debt_kwh * scale))
        max_amount = int(round((debt_kwh + self.config.feasibility_tolerance_kwh + 0.75) * scale))

        # Store one shortest combo per rounded amount. This is the main practical reduction in the
        # candidate-space implementation.
        dp: Dict[int, Tuple[Tuple[Tuple[str, pd.Timestamp], str, float], ...]] = {0: tuple()}
        for key, state, delta in candidates:
            cents = int(round(delta * scale))
            for amount, combo in list(dp.items()):
                new_amount = amount + cents
                if new_amount > max_amount:
                    continue
                new_combo = combo + ((key, state, delta),)
                if new_amount not in dp or len(new_combo) < len(dp[new_amount]):
                    dp[new_amount] = new_combo

        solutions = []
        tol = self.config.feasibility_tolerance_kwh
        for amount, combo in dp.items():
            if amount == 0:
                continue
            residual = abs(amount - target) / scale
            if residual <= tol:
                solutions.append((residual, len(combo), amount / scale, combo))

        return sorted(solutions, key=lambda x: (x[0], x[1]))[: self.config.max_addition_solutions_per_session]

    def complete_with_additions(self, pattern: Dict[Tuple[str, pd.Timestamp], str]) -> List[Dict[Tuple[str, pd.Timestamp], str]]:
        """Complete reductions by adding the removed energy back within each session window."""
        relevant_sessions = {sid for sid, _ in pattern.keys()}

        # Restricted scoring is used here only to identify which adjusted sessions have energy debt.
        _, balances = self.score_pattern(pattern, relevant_sessions)
        if balances is None:
            return []

        debt_sessions = [sid for sid, bal in balances.items() if bal < -self.config.feasibility_tolerance_kwh]
        patterns = [pattern]

        for sid in debt_sessions:
            next_patterns = []
            for pat in patterns:
                _, bal_one = self.score_pattern(pat, {sid})
                if bal_one is None:
                    continue
                balance = bal_one[sid]
                if balance >= -self.config.feasibility_tolerance_kwh:
                    next_patterns.append(pat)
                    continue
                debt = -balance
                adjusted_load = self.adjusted_load_from_pattern(pat)
                additions = self.find_additions_for_session(sid, debt, pat, adjusted_load)
                for _, _, _, combo in additions:
                    new_pat = dict(pat)
                    for key, state, _ in combo:
                        new_pat[key] = state
                    next_patterns.append(new_pat)
            patterns = next_patterns
            if not patterns:
                return []
        return patterns

    def run(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Run candidate generation, full Step 11 CPT scoring, posterior normalisation, and feasibility checks."""
        reduction_patterns = self.enumerate_reduction_patterns()

        completed_patterns: List[Dict[Tuple[str, pd.Timestamp], str]] = []
        for pat in reduction_patterns:
            completed_patterns.extend(self.complete_with_additions(pat))
            if len(completed_patterns) > self.config.max_completed_patterns_to_score:
                warnings.warn(
                    "Reached max_completed_patterns_to_score. Increase the config value if you want a wider candidate set."
                )
                completed_patterns = completed_patterns[: self.config.max_completed_patterns_to_score]
                break

        candidate_rows = []
        posterior_rows = []
        patterns_by_id: Dict[int, Dict[Tuple[str, pd.Timestamp], str]] = {}

        # First score and check every completed candidate pattern.
        for pattern_id, pattern in enumerate(completed_patterns, start=1):
            patterns_by_id[pattern_id] = pattern
            checks = self.check_feasibility(pattern)
            candidate_rows.append({
                "pattern_id": pattern_id,
                "n_adjusted_session_hours": len(pattern),
                "n_scored_flexible_session_hours": len(self.all_flexible_keys),
                "n_implicit_unchanged_session_hours": len(self.all_flexible_keys) - len(pattern),
                **{k: v for k, v in checks.items() if k not in ["balances", "adjusted_loads", "log_probability"]},
                "log_probability": checks["log_probability"],
            })

        candidates = pd.DataFrame(candidate_rows)

        if not candidates.empty:
            admissible_mask = candidates[[
                "target_cap_ok", "energy_preserved_ok", "connection_window_ok", "capacity_bounds_ok", "nonflex_unchanged_ok"
            ]].all(axis=1) & np.isfinite(candidates["log_probability"])
            candidates["admissible"] = admissible_mask

            # Posterior probabilities are normalised only over admissible candidate patterns.
            admissible = candidates[candidates["admissible"]].copy()
            if not admissible.empty:
                logp = admissible["log_probability"].to_numpy(dtype=float)
                probs = np.exp(logp - np.max(logp))
                probs = probs / probs.sum()
                candidates.loc[admissible.index, "posterior_probability"] = probs
                candidates["posterior_probability"] = candidates["posterior_probability"].fillna(0.0)
            else:
                candidates["posterior_probability"] = 0.0
        else:
            candidates["admissible"] = []
            candidates["posterior_probability"] = []

        # Keep the candidate detail table focused on explicit reductions/additions, as in the original
        # output format. Omitted flexible session-hours are still scored as "unchanged" in Step 11, but
        # they are not repeated in this detail table to keep the output readable.
        pattern_detail_rows: List[Dict[str, Any]] = []
        if not candidates.empty:
            status_by_pattern = candidates.set_index("pattern_id")[["admissible", "posterior_probability"]].to_dict("index")
            for pattern_id, pattern in patterns_by_id.items():
                for (sid, ts), f_state in pattern.items():
                    rec = self.row_records[(sid, ts)]
                    delta = delta_from_f_req_state(f_state, rec["eobs"], rec["eidle"])
                    pattern_detail_rows.append({
                        "pattern_id": pattern_id,
                        "session_id": sid,
                        "date_from": ts,
                        "f_req_state": f_state,
                        "delta_kwh": delta,
                        "e_obs_kwh": rec["eobs"],
                        "e_idle_kwh": rec["eidle"],
                        "e_conn_kwh": rec["econn"],
                        "explicitly_assigned_in_candidate": True,
                        "is_adjusted": f_state != "unchanged",
                        "admissible": bool(status_by_pattern[pattern_id]["admissible"]),
                        "posterior_probability": float(status_by_pattern[pattern_id]["posterior_probability"]),
                    })

        details = pd.DataFrame(pattern_detail_rows)

        # Posterior marginal P(f_req_i_t = state | evidence) over the generated admissible candidate set.
        # This loop covers the full F_req domain. Therefore each flexible session-hour receives a complete
        # five-state marginal distribution, including "unchanged" for omitted session-hours.
        posterior_accum: Dict[Tuple[str, pd.Timestamp, str], float] = {}
        if not candidates.empty:
            status_by_pattern = candidates.set_index("pattern_id")[["admissible", "posterior_probability"]].to_dict("index")
            for pattern_id, pattern in patterns_by_id.items():
                if not bool(status_by_pattern[pattern_id]["admissible"]):
                    continue
                pattern_prob = float(status_by_pattern[pattern_id]["posterior_probability"])
                for sid in sorted(self.session_keys):
                    for key in self.session_keys[sid]:
                        state = self.f_state_for_key(pattern, key)
                        posterior_accum[(sid, key[1], state)] = posterior_accum.get((sid, key[1], state), 0.0) + pattern_prob

        for sid in sorted(self.session_keys):
            for key in self.session_keys[sid]:
                total = 0.0
                for state in F_REQ_STATES:
                    total += posterior_accum.get((sid, key[1], state), 0.0)
                for state in F_REQ_STATES:
                    posterior_rows.append({
                        "session_id": sid,
                        "date_from": key[1],
                        "f_req_state": state,
                        "posterior_probability": float(posterior_accum.get((sid, key[1], state), 0.0)),
                        "marginal_total_for_session_hour": float(total),
                    })

        posterior = pd.DataFrame(posterior_rows)
        return candidates, details, posterior

    def check_feasibility(self, pattern: Dict[Tuple[str, pd.Timestamp], str]) -> Dict[str, Any]:
        """Apply the five feasibility checks from Section 6.7."""
        adjusted = self.adjusted_load_from_pattern(pattern)
        target_cap_ok = all(adjusted.get(ts, 0.0) <= self.c90 + 1e-12 for ts in self.target_hours)

        capacity_bounds_ok = True
        connection_window_ok = True
        nonflex_unchanged_ok = True

        # These checks inspect explicit non-unchanged adjustments. Omitted keys are unchanged
        # by construction and therefore have delta_i_t = 0.
        for key, f_state in pattern.items():
            if key not in self.row_records:
                connection_window_ok = False
                continue

            sid, ts = key
            rec = self.row_records[key]
            delta = delta_from_f_req_state(f_state, rec["eobs"], rec["eidle"])
            adjusted_energy = rec["eobs"] + delta

            if not (-self.config.feasibility_tolerance_kwh <= adjusted_energy <= rec["econn"] + self.config.feasibility_tolerance_kwh):
                capacity_bounds_ok = False

            if not rec["is_flexible"]:
                nonflex_unchanged_ok = False

            if delta > 0 and not (self.week_start <= ts < self.week_end):
                connection_window_ok = False

            if key not in self.all_flexible_key_set:
                connection_window_ok = False

        # Final Step 11 score is computed over the full F_req domain, not only changed rows.
        log_probability, balances = self.score_pattern(pattern, relevant_sessions=None)
        if balances is None:
            balances = {}

        energy_preserved_ok = all(abs(b) <= self.config.feasibility_tolerance_kwh for b in balances.values())

        return {
            "target_cap_ok": target_cap_ok,
            "energy_preserved_ok": energy_preserved_ok,
            "connection_window_ok": connection_window_ok,
            "capacity_bounds_ok": capacity_bounds_ok,
            "nonflex_unchanged_ok": nonflex_unchanged_ok,
            "log_probability": log_probability,
            "balances": balances,
            "adjusted_loads": adjusted,
        }


def build_mapping_table() -> pd.DataFrame:
    """Thesis-to-code mapping table."""
    return pd.DataFrame([
        {"thesis_symbol": "i, I", "python_variable": "session_id, sessions", "role": "identifier/scope", "dataset": "Dataset 1/3/4", "method": "filter and merge"},
        {"thesis_symbol": "t, T", "python_variable": "date_from, target_hours", "role": "hourly time index", "dataset": "Dataset 4", "method": "timestamp parsing"},
        {"thesis_symbol": "W", "python_variable": "target_hours", "role": "target window", "dataset": "Dataset 4", "method": "weekday 16:00-21:00 mask"},
        {"thesis_symbol": "e_obs_i_t", "python_variable": "energy_charged_i, e_obs_state", "role": "input node", "dataset": "Dataset 4", "method": "Appendix A deterministic CPT"},
        {"thesis_symbol": "e_idle_i_t", "python_variable": "energy_idle_i, e_idle_state", "role": "input node", "dataset": "Dataset 4", "method": "Appendix A deterministic CPT"},
        {"thesis_symbol": "e_conn_i_t", "python_variable": "energy_connected_i, e_conn_state", "role": "input node", "dataset": "Dataset 4", "method": "Appendix A deterministic CPT"},
        {"thesis_symbol": "C90", "python_variable": "c90", "role": "target cap", "dataset": "Dataset 4", "method": "90th percentile of selected week W loads"},
        {"thesis_symbol": "f_req_i_t", "python_variable": "f_req_state", "role": "hidden/inferred node", "dataset": "not directly observed", "method": "Appendix A CPT + candidate posterior"},
        {"thesis_symbol": "delta_i_t", "python_variable": "delta_kwh", "role": "deterministic support", "dataset": "computed", "method": "Equation 6.23 midpoint mapping"},
        {"thesis_symbol": "B_i_t", "python_variable": "balance", "role": "external deterministic support", "dataset": "computed", "method": "sequential balance update"},
        {"thesis_symbol": "tilde_L_t", "python_variable": "adjusted_load", "role": "output node", "dataset": "computed", "method": "L_obs + sum(delta)"},
        {"thesis_symbol": "X_t", "python_variable": "X_t", "role": "target evidence", "dataset": "computed", "method": "0 if within cap, 1 if exceeds cap"},
    ])


def make_input_summary(data: Dict[str, pd.DataFrame], sessions: pd.DataFrame, hourly: pd.DataFrame, target_loads: pd.Series, c90: float, exceed_hours: List[pd.Timestamp], week_start: pd.Timestamp, week_end: pd.Timestamp) -> pd.DataFrame:
    """Cleaned and validated input summary."""
    return pd.DataFrame([
        {"item": "selected_location", "value": CONFIG.location},
        {"item": "selected_target_week_start", "value": str(week_start)},
        {"item": "selected_target_week_end", "value": str(week_end)},
        {"item": "sessions_in_scope", "value": len(sessions)},
        {"item": "flexible_sessions_in_scope", "value": int(sessions["is_flexible"].sum())},
        {"item": "nonflexible_sessions_in_scope", "value": int((~sessions["is_flexible"]).sum())},
        {"item": "hourly_rows_in_scope", "value": len(hourly)},
        {"item": "target_weekday_evening_hours", "value": len(target_loads)},
        {"item": "computed_C90_kwh", "value": c90},
        {"item": "exceeded_target_hours_before_adjustment", "value": len(exceed_hours)},
        {"item": "exceeded_target_hour_list", "value": "; ".join(map(str, exceed_hours))},
    ])


In [29]:
# =============================================================================
# 11. VISUALISATIONS
# =============================================================================

def save_visualisations(output_dir: Path, candidates: pd.DataFrame, details: pd.DataFrame, posterior: pd.DataFrame, target_loads: pd.Series, c90: float, inference_engine: FactorBasedWeekInference) -> None:
    """
    Create thesis-facing visualisations from the corrected posterior over admissible patterns.

    This function only changes the result visualisation layer. It does not change the inference
    procedure, candidate generation, Step 11 scoring, posterior normalisation, or feasibility checks.

    The previous broad f_req marginal plots were technically valid, but they were poor thesis figures
    because they were dominated by the many unchanged flexible session-hours. The figures below focus
    on the admissible candidate patterns and on the delta_i_t energy shifts, which makes it clearer how
    the inferred flexibility-use patterns reduce peak hours and add the removed energy back elsewhere.
    """
    plots_dir = output_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    # Remove older broad-domain marginal plots if they exist, because they can make it look as if
    # almost nothing is shifted. The corrected thesis-facing figures below focus on actual delta_i_t shifts.
    for obsolete_plot in [
        "f_req_state_probability_barplot.png",
        "posterior_heatmap_flexible_session_hours.png",
    ]:
        obsolete_path = plots_dir / obsolete_plot
        if obsolete_path.exists():
            obsolete_path.unlink()

    # Plot 1: observed target-window load and C90.
    # This shows the backward-inference starting point: which weekday-evening hours violate the cap
    # before any flexibility-use pattern is applied.
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(target_loads.index, target_loads.values, marker="o", label="Observed $L^{obs}_t$")
    ax.axhline(c90, linestyle="--", label="$C_{90}$ cap")
    ax.set_title("Observed weekday-evening charging load before inference")
    ax.set_xlabel("Target-window hour")
    ax.set_ylabel("kWh per hour")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(plots_dir / "observed_load_vs_c90.png", dpi=200)
    plt.close(fig)

    if candidates.empty or details.empty:
        return

    admissible = candidates[candidates["admissible"]].copy()
    if admissible.empty:
        return

    # Use only admissible patterns for thesis result figures, because only these patterns satisfy all
    # target-state and physical feasibility constraints from Chapter 6.
    pattern_weights = admissible.set_index("pattern_id")["posterior_probability"]
    admissible_details = details[details["pattern_id"].isin(pattern_weights.index)].copy()
    admissible_details["date_from"] = pd.to_datetime(admissible_details["date_from"])
    admissible_details["posterior_weight"] = admissible_details["pattern_id"].map(pattern_weights).astype(float)
    admissible_details["weighted_delta_kwh"] = admissible_details["delta_kwh"] * admissible_details["posterior_weight"]

    # Build the observed aggregate load for all hourly rows in the selected implementation scope.
    # This is broader than the target window and therefore also shows where energy is added back.
    hourly_scope = inference_engine.hourly.copy()
    hourly_scope["date_from"] = pd.to_datetime(hourly_scope["date_from"])
    observed_all_hours = hourly_scope.groupby("date_from")["energy_charged_i"].sum().sort_index()

    # Posterior-expected net energy shift per hour:
    # E[Delta_t] = sum_k P(F_req^(k)|evidence,constraints) * sum_i delta_i,t^(k)
    expected_delta_by_hour = (
        admissible_details.groupby("date_from")["weighted_delta_kwh"].sum()
        .reindex(observed_all_hours.index, fill_value=0.0)
        .sort_index()
    )
    posterior_expected_adjusted_load = observed_all_hours + expected_delta_by_hour

    # Plot 2: posterior probabilities of top admissible candidate patterns.
    # This communicates that the model output is a ranked distribution, not one optimal schedule.
    top = admissible.sort_values("posterior_probability", ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(top["pattern_id"].astype(str), top["posterior_probability"])
    ax.set_title("Top admissible flexibility-use patterns")
    ax.set_xlabel("Pattern ID")
    ax.set_ylabel("Posterior probability within admissible set")
    fig.tight_layout()
    fig.savefig(plots_dir / "top_pattern_probabilities.png", dpi=200)
    plt.close(fig)

    # Plot 3: observed versus posterior-expected adjusted load in the target window only.
    # C90 is only a target-window cap, so this plot deliberately focuses on weekday-evening
    # target hours instead of all connected hours in the selected scope.
    target_index = pd.DatetimeIndex(target_loads.index)
    expected_delta_target = expected_delta_by_hour.reindex(target_index, fill_value=0.0)
    posterior_expected_target_load = target_loads + expected_delta_target

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(target_index, target_loads.values, marker="o", label="Observed target-window load")
    ax.plot(
        target_index,
        posterior_expected_target_load.values,
        marker="o",
        label="Posterior-expected adjusted target-window load",
    )
    ax.axhline(c90, linestyle="--", label="$C_{90}$ cap")
    ax.set_title("Observed versus posterior-expected adjusted load in target hours")
    ax.set_xlabel("Target-window hour")
    ax.set_ylabel("kWh per hour")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(plots_dir / "observed_vs_posterior_expected_adjusted_load.png", dpi=200)
    plt.close(fig)

    # Plot 4: posterior-expected net shift by hour.
    # Negative bars show expected peak-hour reductions. Positive bars show where the removed energy is
    # expected to be added back. This is the clearest plot for explaining how energy is shifted.
    nonzero_expected_delta = expected_delta_by_hour[expected_delta_by_hour.abs() > 1e-9]
    if not nonzero_expected_delta.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(nonzero_expected_delta.index, nonzero_expected_delta.values, width=0.03)
        ax.axhline(0.0, linewidth=1)
        ax.set_title("Posterior-expected charging-energy shift by hour")
        ax.set_xlabel("Hour")
        ax.set_ylabel(r"Expected net shift $E[\Delta_t]$ [kWh]")
        fig.autofmt_xdate()
        fig.tight_layout()
        fig.savefig(plots_dir / "posterior_expected_energy_shift_by_hour.png", dpi=200)
        plt.close(fig)

    # Plot 5: observed load versus adjusted load for the highest-posterior admissible pattern.
    # This gives one concrete example of an admissible inferred pattern, which is easier to explain
    # than the posterior expectation alone.
    best_pattern_id = int(top.iloc[0]["pattern_id"])
    best_detail = admissible_details[(admissible_details["pattern_id"].eq(best_pattern_id)) & (admissible_details["is_adjusted"])].copy()
    pattern = {
        (str(r["session_id"]), pd.Timestamp(r["date_from"])): r["f_req_state"]
        for _, r in best_detail.iterrows()
    }
    # For the top pattern, compute the adjusted load directly from its explicit delta_i_t values.
    # Unassigned session-hours have delta_i_t = 0 and therefore leave the observed load unchanged.
    # The plotted load is restricted to target-window hours, because C90 is evaluated there.
    best_delta_target = best_detail.groupby("date_from")["delta_kwh"].sum().reindex(target_index, fill_value=0.0)
    best_adjusted_target_load = target_loads + best_delta_target

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(target_index, target_loads.values, marker="o", label="Observed target-window load")
    ax.plot(target_index, best_adjusted_target_load.values, marker="o", label="Adjusted target-window load, top pattern")
    ax.axhline(c90, linestyle="--", label="$C_{90}$ cap")
    ax.set_title(f"Observed versus adjusted target-window load for top pattern {best_pattern_id}")
    ax.set_xlabel("Target-window hour")
    ax.set_ylabel("kWh per hour")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(plots_dir / "observed_vs_adjusted_best_pattern.png", dpi=200)
    plt.close(fig)

    # Plot 6: concrete net shift by hour for the highest-posterior admissible pattern.
    # This shows exactly which hours are reduced and which hours receive the shifted energy in the
    # most likely admissible pattern.
    best_delta_by_hour = best_detail.groupby("date_from")["delta_kwh"].sum().sort_index()
    best_delta_by_hour = best_delta_by_hour[best_delta_by_hour.abs() > 1e-9]
    if not best_delta_by_hour.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.bar(best_delta_by_hour.index, best_delta_by_hour.values, width=0.03)
        ax.axhline(0.0, linewidth=1)
        ax.set_title(f"Charging-energy shift by hour for top admissible pattern {best_pattern_id}")
        ax.set_xlabel("Hour")
        ax.set_ylabel(r"Net shift $\Delta_t$ [kWh]")
        fig.autofmt_xdate()
        fig.tight_layout()
        fig.savefig(plots_dir / "top_pattern_energy_shift_by_hour.png", dpi=200)
        plt.close(fig)

    # Plot 7: shifted energy by session for the highest-posterior admissible pattern.
    # This supports the thesis interpretation that shifting happens within flexible sessions and that
    # removed energy is added back to the same sessions.
    if not best_detail.empty:
        session_shift = (
            best_detail.assign(
                reduced_kwh=lambda df: np.where(df["delta_kwh"] < 0, -df["delta_kwh"], 0.0),
                added_kwh=lambda df: np.where(df["delta_kwh"] > 0, df["delta_kwh"], 0.0),
            )
            .groupby("session_id")[["reduced_kwh", "added_kwh"]]
            .sum()
        )
        session_shift = session_shift[(session_shift["reduced_kwh"] > 1e-9) | (session_shift["added_kwh"] > 1e-9)]
        if not session_shift.empty:
            # Plot reductions as negative values and additions as positive values.
            # This makes the within-session energy preservation logic easier to see visually.
            session_shift_signed = pd.DataFrame({
                "reduced_kwh": -session_shift["reduced_kwh"],
                "added_kwh": session_shift["added_kwh"],
            })
            y_pos = np.arange(len(session_shift_signed))
            bar_height = 0.35
            fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(session_shift_signed))))
            ax.barh(y_pos - bar_height / 2, session_shift_signed["reduced_kwh"].values, height=bar_height, label="Reduced energy")
            ax.barh(y_pos + bar_height / 2, session_shift_signed["added_kwh"].values, height=bar_height, label="Added energy")
            ax.axvline(0.0, linewidth=1)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(session_shift_signed.index.astype(str))
            ax.set_title(f"Reduced and added energy by session, top pattern {best_pattern_id}")
            ax.set_xlabel("kWh; negative = reduced, positive = added")
            ax.set_ylabel("Session ID")
            ax.legend()
            fig.tight_layout()
            fig.savefig(plots_dir / "top_pattern_shifted_energy_by_session.png", dpi=200)
            plt.close(fig)

In [31]:
# =============================================================================
# 11A. FULL TARGET-WEEK LOAD PROFILE VISUALISATION
# =============================================================================

_base_save_visualisations = getattr(save_visualisations, "_base_visualisation_function", save_visualisations)


def save_full_target_week_load_profile(
    output_dir: Path,
    candidates: pd.DataFrame,
    details: pd.DataFrame,
    inference_engine: FactorBasedWeekInference,
) -> None:
    """Save a full target-week before/after aggregate load plot with shaded target windows."""
    plots_dir = output_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    full_week_index = pd.date_range(
        start=inference_engine.week_start,
        end=inference_engine.week_end,
        freq="h",
        inclusive="left",
    )

    hourly_scope = inference_engine.hourly.copy()
    hourly_scope["date_from"] = pd.to_datetime(hourly_scope["date_from"])
    observed_full_week = (
        hourly_scope.groupby("date_from")["energy_charged_i"].sum()
        .reindex(full_week_index, fill_value=0.0)
        .sort_index()
    )

    expected_delta_full_week = pd.Series(0.0, index=full_week_index)
    if not candidates.empty and not details.empty and "admissible" in candidates.columns and candidates["admissible"].any():
        admissible = candidates[candidates["admissible"]].copy()
        pattern_weights = admissible.set_index("pattern_id")["posterior_probability"]
        admissible_details = details[details["pattern_id"].isin(pattern_weights.index)].copy()
        admissible_details["date_from"] = pd.to_datetime(admissible_details["date_from"])
        admissible_details["posterior_weight"] = admissible_details["pattern_id"].map(pattern_weights).astype(float)
        admissible_details["weighted_delta_kwh"] = admissible_details["delta_kwh"] * admissible_details["posterior_weight"]
        expected_delta_full_week = (
            admissible_details.groupby("date_from")["weighted_delta_kwh"].sum()
            .reindex(full_week_index, fill_value=0.0)
            .sort_index()
        )

    adjusted_full_week = observed_full_week + expected_delta_full_week

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(observed_full_week.index, observed_full_week.values, color="tab:blue", label="Aggregate load before adjustments")
    ax.plot(adjusted_full_week.index, adjusted_full_week.values, color="tab:orange", label="Aggregate load after adjustments")
    ax.axhline(inference_engine.c90, linestyle=":", color="black", label=r"$C^{week}_{90}$")

    shaded_label_used = False
    last_day = (inference_engine.week_end - pd.Timedelta(hours=1)).normalize()
    for day in pd.date_range(inference_engine.week_start.normalize(), last_day, freq="D"):
        if day.weekday() < 5:
            shade_start = day + pd.Timedelta(hours=inference_engine.config.target_start_hour)
            shade_end = day + pd.Timedelta(hours=inference_engine.config.target_end_hour_exclusive)
            if shade_end <= inference_engine.week_start or shade_start >= inference_engine.week_end:
                continue
            ax.axvspan(
                max(shade_start, inference_engine.week_start),
                min(shade_end, inference_engine.week_end),
                color="limegreen",
                alpha=0.25,  # Increase alpha for darker shading; decrease alpha for lighter shading.
                label="Weekday-evening target window" if not shaded_label_used else None,
            )
            shaded_label_used = True

    locator = plt.matplotlib.dates.AutoDateLocator(minticks=6, maxticks=10)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.ConciseDateFormatter(locator))
    ax.set_xlim(inference_engine.week_start, inference_engine.week_end)
    ax.set_title("Aggregate charging load profile before and after adjustments")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Aggregate load (kWh per hour)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(plots_dir / "full_target_week_observed_vs_adjusted_load.png", dpi=200)
    plt.close(fig)


def save_visualisations(output_dir: Path, candidates: pd.DataFrame, details: pd.DataFrame, posterior: pd.DataFrame, target_loads: pd.Series, c90: float, inference_engine: FactorBasedWeekInference) -> None:
    _base_save_visualisations(output_dir, candidates, details, posterior, target_loads, c90, inference_engine)
    save_full_target_week_load_profile(output_dir, candidates, details, inference_engine)


save_visualisations._base_visualisation_function = _base_save_visualisations


In [33]:
from collections import Counter
import itertools
import pandas as pd


# =============================================================================
# Candidate-space counting check:
# reduction patterns -> debt-session groups -> upper-bound completed patterns
# =============================================================================

# Load and preprocess the selected target period, using the notebook's existing logic.
data = load_all_data(FILE_PATHS)
parse_time_columns(data)
validate_hourly_time_format(data["dataset4"])

sessions, hourly, target_hours, week_start, week_end = select_target_week_scope(data, CONFIG)
hourly = add_discrete_state_columns(hourly)

target_loads, c90, exceed_hours = compute_observed_target_loads(
    hourly, target_hours, week_start, week_end
)

# Rebuild the inference engine, but do not call inference_engine.run().
inference_engine = FactorBasedWeekInference(
    hourly=hourly,
    target_hours=target_hours,
    week_start=week_start,
    week_end=week_end,
    c90=c90,
    target_loads=target_loads,
    exceed_hours=exceed_hours,
    config=CONFIG,
)


# -----------------------------------------------------------------------------
# Step 1: Count feasible reduction patterns per exceeded target hour
# -----------------------------------------------------------------------------

hour_pattern_counts = []

for ts in inference_engine.exceed_hours:
    needed_reduction = float(inference_engine.target_loads.loc[ts] - inference_engine.c90)

    active = inference_engine.hourly[
        (inference_engine.hourly["date_from"].eq(ts))
        & (inference_engine.hourly["is_flexible"])
        & (inference_engine.hourly["energy_charged_i"].gt(0.0))
    ]

    active_keys = [
        (str(r["session_id"]), pd.Timestamp(r["date_from"]))
        for _, r in active.iterrows()
    ]

    n_feasible_patterns = 0

    for combo in itertools.product(
        *[inference_engine.reduction_options_for_key(k) for k in active_keys]
    ):
        total_reduction = -sum(delta for _, delta in combo)

        if total_reduction >= needed_reduction - 1e-12:
            n_feasible_patterns += 1

    hour_pattern_counts.append({
        "exceeded_hour": ts,
        "active_flexible_session_hours": len(active_keys),
        "needed_reduction_kwh": needed_reduction,
        "feasible_reduction_patterns": n_feasible_patterns,
    })

hour_pattern_table = pd.DataFrame(hour_pattern_counts)

combined_base_reduction_patterns = int(
    hour_pattern_table["feasible_reduction_patterns"].prod()
)


# -----------------------------------------------------------------------------
# Step 2: Enumerate all combined base reduction patterns
# -----------------------------------------------------------------------------

reduction_patterns = inference_engine.enumerate_reduction_patterns()


# -----------------------------------------------------------------------------
# Step 3: For each combined base reduction pattern, count unique debt sessions
# -----------------------------------------------------------------------------

debt_session_counter = Counter()

for pattern in reduction_patterns:
    relevant_sessions = {sid for sid, _ in pattern.keys()}

    _, balances = inference_engine.score_pattern(
        pattern,
        relevant_sessions=relevant_sessions,
    )

    if balances is None:
        continue

    debt_sessions = [
        sid
        for sid, balance in balances.items()
        if balance < -inference_engine.config.feasibility_tolerance_kwh
    ]

    debt_session_counter[len(debt_sessions)] += 1


# -----------------------------------------------------------------------------
# Step 4: Multiply each row by 5^n, where n = number of debt sessions
# -----------------------------------------------------------------------------

rows = []

for n_debt_sessions in sorted(debt_session_counter):
    n_base_patterns = debt_session_counter[n_debt_sessions]

    max_completions_per_pattern = (
        inference_engine.config.max_addition_solutions_per_session
        ** n_debt_sessions
    )

    row_upper_bound = n_base_patterns * max_completions_per_pattern

    rows.append({
        "debt_sessions": n_debt_sessions,
        "base_reduction_patterns": n_base_patterns,
        "max_completions_per_pattern": max_completions_per_pattern,
        "row_upper_bound_completed_patterns": row_upper_bound,
    })

candidate_space_table = pd.DataFrame(rows)

total_upper_bound_completed_patterns = int(
    candidate_space_table["row_upper_bound_completed_patterns"].sum()
)


# -----------------------------------------------------------------------------
# Step 5: Print results clearly
# -----------------------------------------------------------------------------

print("Selected target period")
print("----------------------")
print(f"week_start: {week_start}")
print(f"week_end:   {week_end}")
print(f"duration:   {week_end - week_start}")
print(f"C90 cap:    {c90:.6f} kWh")
print()

print("Feasible reduction patterns per exceeded target hour")
print("----------------------------------------------------")
print(hour_pattern_table.to_string(index=False))
print()

print("Combined base reduction patterns")
print("--------------------------------")
print(
    " × ".join(
        f"{int(x):,}"
        for x in hour_pattern_table["feasible_reduction_patterns"]
    )
    + f" = {combined_base_reduction_patterns:,}"
)
print()

print("Debt-session grouping and upper-bound completed patterns")
print("--------------------------------------------------------")
print(candidate_space_table.to_string(index=False))
print()

print("Total upper-bound completed candidate patterns")
print("----------------------------------------------")
print(f"{total_upper_bound_completed_patterns:,}")
print(f"≈ {total_upper_bound_completed_patterns / 1_000_000_000:.1f} billion")


# Optional: save the tables to the notebook output directory
hour_pattern_table.to_csv(
    CONFIG.output_dir / "candidate_space_exceeded_hour_counts.csv",
    index=False,
)

candidate_space_table.to_csv(
    CONFIG.output_dir / "candidate_space_upper_bound_table.csv",
    index=False,
)

Selected target period
----------------------
week_start: 2019-06-17 00:00:00
week_end:   2019-06-22 00:00:00
duration:   5 days 00:00:00
C90 cap:    23.064494 kWh

Feasible reduction patterns per exceeded target hour
----------------------------------------------------
      exceeded_hour  active_flexible_session_hours  needed_reduction_kwh  feasible_reduction_patterns
2019-06-18 20:00:00                              5              1.065860                          237
2019-06-18 21:00:00                              4              5.170031                           57
2019-06-20 18:00:00                              3              1.426346                           23

Combined base reduction patterns
--------------------------------
237 × 57 × 23 = 310,707

Debt-session grouping and upper-bound completed patterns
--------------------------------------------------------
 debt_sessions  base_reduction_patterns  max_completions_per_pattern  row_upper_bound_completed_patterns
          